In [1]:
!pip install -U datasets fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 22.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2026.6.0 which is incompatible.


In [2]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 110.4 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1


In [9]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

In [12]:
dataset = load_dataset('stanfordnlp/imdb')
small_train = dataset['train'].shuffle(seed=42).select([i for i in list(range(3000))])
small_test = dataset['test'].shuffle(seed=42).select([i for i in list(range(300))])

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [13]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=256)

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_test.map(preprocess_function, batched=True)
#


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

In [14]:
model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    logging_steps=10,
    eval_strategy="steps",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

Step,Training Loss,Validation Loss
10,0.687344,0.681276
20,0.658334,0.595961
30,0.554161,0.482332
40,0.417303,0.422219
50,0.451949,0.463738
60,0.470857,0.449264
70,0.451568,0.417322
80,0.323559,0.376733
90,0.425861,0.386244
100,0.281937,0.343614


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=564, training_loss=0.23673885965600927, metrics={'train_runtime': 333.123, 'train_samples_per_second': 27.017, 'train_steps_per_second': 1.693, 'total_flos': 596103293952000.0, 'train_loss': 0.23673885965600927, 'epoch': 3.0})

In [18]:
texts = [
    'I love this movie',
    'This movie was absolutely awful',
    'Just ok movie, not so bad',
    'Movie just bad, or has advantages'
]

In [22]:
import torch
import torch.nn.functional as F

# Dynamically select GPU if available, else CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Move model to the same device
model = model.to(device)

# Preprocess and move inputs to the selected device
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=256).to(device)

with torch.no_grad():
    logits = model(**inputs)
    probs = F.softmax(logits.logits, dim=1)
    predictions = torch.argmax(probs, dim=1)

label_map = {0: 'negative', 1: 'positive'}

# Correctly zip and iterate using the 'probs' and 'predictions' tensors
for text, pred, prob in zip(texts, predictions, probs):
    print(f"Text: '{text}' -> Sentiment: {label_map[pred.item()]} (Confidence: {prob[pred.item()].item():.4f})")

Text: 'I love this movie' -> Sentiment: positive (Confidence: 0.9946)
Text: 'This movie was absolutely awful' -> Sentiment: negative (Confidence: 0.9966)
Text: 'Just ok movie, not so bad' -> Sentiment: negative (Confidence: 0.9949)
Text: 'Movie just bad, or has advantages' -> Sentiment: negative (Confidence: 0.9972)
